In [29]:
import requests
from pathlib import Path

import duckdb

from to_gpx import to_gpx

In [ ]:
DATA_BASE_PATH = Path("./data").resolve().absolute()
GRAPHHOPPER_BASE_URL = "http://localhost:8989"
GPS_ACCURACY = 50

In [31]:
gps_data_path = DATA_BASE_PATH / "gps_data.parquet"

In [32]:
df = duckdb.query(f"SELECT * FROM '{gps_data_path}'").to_df()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7531 entries, 0 to 7530
Data columns (total 3 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   recorded_timestamp  7531 non-null   datetime64[us]
 1   lon                 7531 non-null   float64       
 2   lat                 7531 non-null   float64       
dtypes: datetime64[us](1), float64(2)
memory usage: 176.6 KB


In [33]:
df.head()

,recorded_timestamp,lon,lat
0,2009-01-17 20:27:37,-122.107083,47.667483
1,2009-01-17 20:27:38,-122.107067,47.667500
2,2009-01-17 20:27:39,-122.107067,47.667500
3,2009-01-17 20:27:40,-122.107033,47.667517
4,2009-01-17 20:27:41,-122.106983,47.667533


In [34]:
points = df[["lon", "lat", "recorded_timestamp"]].to_numpy()

gpx_path = to_gpx(points, DATA_BASE_PATH / "gps.gpx")

In [ ]:
def request_map_matching(point_gpx_path: Path, output_path: Path):
    url = f"{GRAPHHOPPER_BASE_URL}/match?profile=car&gps_accuracy={GPS_ACCURACY}&type=json"
    headers = {
        "Content-Type": "application/gpx+xml",
    }

    with open(point_gpx_path, "rb") as f:
        body = f.read()

    req = requests.post(
        url,
        headers=headers,
        data=body,
    )

    req.raise_for_status()

    with open(output_path, "wb") as f:
        f.write(req.content)

    print(f"Map matching result saved to {output_path}")

In [36]:
request_map_matching(
    point_gpx_path=gpx_path,
    output_path=DATA_BASE_PATH / "map_matched.json"
)

Map matching result saved to /home/jose_edsouza/Documentos/Faculdade/TCC/repo/dataset/newson-krumm/data/map_matched.json
